# Create a dedicated virtual environment inside project folder

In [1]:
import sys
print(sys.executable)

c:\Users\HP\OneDrive\Desktop\NLP lab\.venv\Scripts\python.exe


# NLP Lab 1 & 2 — Integrated Assignment
### Text Preprocessing, Spelling Correction, Text Representation, and N-Gram Language Modeling

This notebook merges **every concept taught in Lab 1** (text preprocessing and spelling
correction) and **Lab 2** (Bag of Words, TF-IDF, and probabilistic N-gram language modeling)
into a single, self-contained, educational pipeline.

Each section contains a short theory recap in Markdown followed by a working, commented
implementation. All intermediate outputs are printed so the full pipeline is transparent
from raw text all the way to sentence generation.

## Objective

By the end of this notebook you will have implemented, from first principles and using
standard libraries, the following pipeline:

1. **Text Preprocessing** — regex cleaning, HTML removal, number/special-character removal,
   whitespace normalization, tokenization, stop-word removal, stemming, lemmatization.
2. **Bag of Words (BoW)** vector representation of a corpus.
3. **TF-IDF** vector representation of a corpus.
4. **Levenshtein Edit Distance** and a dictionary-based **spelling corrector**.
5. **Sentence-boundary-aware corpus construction** using `<s>` / `</s>` tokens.
6. A **fully dynamic N-Gram Language Model** (works for any N = 1, 2, 3, 4, ...).
7. **Count tables** and **MLE probability tables** for the trained model.
8. **Shannon's Guessing Game** — next-word prediction.
9. **Probabilistic sentence generation** from the trained model.

All outputs are displayed using `pandas` DataFrames wherever tabular data is produced.

## Required Libraries

We use only standard NLP / data-science Python libraries:

* `re` — regular expressions for text cleaning.
* `nltk` — tokenization, stop words, stemming (Porter), lemmatization (WordNet).
* `numpy` — dynamic-programming matrix for edit distance, weighted random sampling.
* `pandas` — tabular display of every intermediate result.
* `collections.defaultdict` / `Counter` — efficient N-gram counting.
* `sklearn.feature_extraction.text` — `CountVectorizer` (BoW) and `TfidfVectorizer` (TF-IDF).

Run the cell below once to install/download everything needed.

In [2]:
# Install (uncomment if running in a fresh environment)
# %pip install nltk scikit-learn pandas numpy

import re
import math
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Download required NLTK resources (only needs to run once per environment)
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("All libraries imported and NLTK resources downloaded successfully!")

All libraries imported and NLTK resources downloaded successfully!


## 4. Corpus

We define our own English corpus below. Every sentence is explicitly wrapped with
sentence-boundary tokens `<s>` (start) and `</s>` (end), as required for N-gram modeling
(these boundaries let the model learn what words legally start/end a sentence).

The corpus is a small collection of simple declarative sentences on a common theme
(animals and nature), which keeps the vocabulary compact enough that N-gram probability
tables stay easy to read while still being large enough (20 sentences) to produce
non-trivial statistics.

In [3]:
# Raw corpus (no boundary tokens yet) - used for the Preprocessing / BoW / TF-IDF sections
raw_corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "A lazy dog sleeps in the warm sun all afternoon.",
    "Foxes are quick, clever, and highly adaptable animals.",
    "The dog barks loudly whenever a stranger walks by.",
    "Cats and dogs often become the best of friends.",
    "The clever fox outsmarted the farmer once again.",
    "Birds sing sweetly in the early morning light.",
    "A brown bear wandered slowly through the quiet forest.",
    "The forest was calm, green, and full of life.",
    "Rivers flow gently through the deep, ancient forest.",
    "Mountains rise high above the sleepy little village.",
    "The village children played happily near the river.",
    "Rain fell softly on the old wooden rooftops.",
    "Farmers wake early to tend their growing crops.",
    "The crops grew tall under the summer sun.",
    "Wolves howl at night beneath a bright full moon.",
    "The moon lit the quiet valley with silver light.",
    "A gentle breeze moved slowly across the open field.",
    "The field was covered in golden autumn leaves.",
    "Autumn leaves fall gently onto the cold, hard ground.",
]

print(f"Total sentences in raw corpus: {len(raw_corpus)}\n")
for i, s in enumerate(raw_corpus, 1):
    print(f"{i:2d}. {s}")

Total sentences in raw corpus: 20

 1. The quick brown fox jumps over the lazy dog.
 2. A lazy dog sleeps in the warm sun all afternoon.
 3. Foxes are quick, clever, and highly adaptable animals.
 4. The dog barks loudly whenever a stranger walks by.
 5. Cats and dogs often become the best of friends.
 6. The clever fox outsmarted the farmer once again.
 7. Birds sing sweetly in the early morning light.
 8. A brown bear wandered slowly through the quiet forest.
 9. The forest was calm, green, and full of life.
10. Rivers flow gently through the deep, ancient forest.
11. Mountains rise high above the sleepy little village.
12. The village children played happily near the river.
13. Rain fell softly on the old wooden rooftops.
14. Farmers wake early to tend their growing crops.
15. The crops grew tall under the summer sun.
16. Wolves howl at night beneath a bright full moon.
17. The moon lit the quiet valley with silver light.
18. A gentle breeze moved slowly across the open field.
19. T

## 5. Text Preprocessing (Lab 1)

Raw text is noisy. Before any statistical model can use it, we must clean and normalize it.
Below we implement — and print the output of — **every** preprocessing stage taught in Lab 1,
applied step by step to a single example sentence so the transformation at each stage is
clearly visible, and then as a reusable pipeline applied to the whole corpus.

**Stages:**
1. Raw text
2. Lowercasing
3. HTML tag removal (regex)
4. Removal of numbers and special characters (regex)
5. Removal of extra whitespace (regex)
6. Tokenization
7. Stop-word removal
8. Stemming (Porter's Algorithm)
9. Lemmatization (WordNet)

In [4]:
# --- Step-by-step demonstration on one messy example sentence ---
raw_text = "Hello!!! Welcome to the NLP lab in 2026. This text has <br> HTML tags, numbers like 123, and symbols #$%^."
print("1. Raw text            :", raw_text)

# 2. Lowercasing
lower_text = raw_text.lower()
print("2. Lowercased           :", lower_text)

# 3. HTML tag removal
no_html = re.sub(r'<[^>]+>', '', lower_text)
print("3. HTML tags removed    :", no_html)

# 4. Remove numbers and special characters (keep only letters and spaces)
no_special = re.sub(r'[^a-z\s]', '', no_html)
print("4. Numbers/symbols removed:", no_special)

# 5. Remove extra whitespace
no_extra_space = re.sub(r'\s+', ' ', no_special).strip()
print("5. Extra spaces removed :", no_extra_space)

# 6. Tokenization
tokens = word_tokenize(no_extra_space)
print("6. Tokens               :", tokens)

# 7. Stop-word removal
stop_words = set(stopwords.words('english'))
no_stopwords = [w for w in tokens if w not in stop_words]
print("7. After stop-word removal:", no_stopwords)

# 8. Stemming (Porter's Algorithm)
stemmer = PorterStemmer()
stemmed = [stemmer.stem(w) for w in no_stopwords]
print("8. Stemmed              :", stemmed)

# 9. Lemmatization (WordNet)
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w, pos='v') for w in no_stopwords]
print("9. Lemmatized           :", lemmatized)

1. Raw text            : Hello!!! Welcome to the NLP lab in 2026. This text has <br> HTML tags, numbers like 123, and symbols #$%^.
2. Lowercased           : hello!!! welcome to the nlp lab in 2026. this text has <br> html tags, numbers like 123, and symbols #$%^.
3. HTML tags removed    : hello!!! welcome to the nlp lab in 2026. this text has  html tags, numbers like 123, and symbols #$%^.
4. Numbers/symbols removed: hello welcome to the nlp lab in  this text has  html tags numbers like  and symbols 
5. Extra spaces removed : hello welcome to the nlp lab in this text has html tags numbers like and symbols
6. Tokens               : ['hello', 'welcome', 'to', 'the', 'nlp', 'lab', 'in', 'this', 'text', 'has', 'html', 'tags', 'numbers', 'like', 'and', 'symbols']
7. After stop-word removal: ['hello', 'welcome', 'nlp', 'lab', 'text', 'html', 'tags', 'numbers', 'like', 'symbols']
8. Stemmed              : ['hello', 'welcom', 'nlp', 'lab', 'text', 'html', 'tag', 'number', 'like', 'symbol']
9.

### Reusable Preprocessing Function

We now wrap every stage above into a single reusable function, `full_preprocess()`, and
apply it across the whole corpus. This function is reused throughout the rest of the
notebook (BoW, TF-IDF, N-gram training).

In [5]:
def full_preprocess(text, remove_stopwords=True, use_stemming=False, use_lemmatization=True):
    """
    Applies the complete Lab 1 preprocessing suite to a piece of raw text:
    lowercasing -> HTML removal -> special-character/number removal ->
    whitespace normalization -> tokenization -> [stop-word removal] ->
    [stemming] -> [lemmatization].

    Returns a list of cleaned tokens.
    """
    text = text.lower()                                  # Lowercasing
    text = re.sub(r'<[^>]+>', '', text)                   # HTML tag removal
    text = re.sub(r'[^a-z\s]', '', text)                  # Remove numbers & special chars
    text = re.sub(r'\s+', ' ', text).strip()              # Collapse extra whitespace

    tokens = word_tokenize(text)                          # Tokenization

    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [w for w in tokens if w not in stop_words]

    if use_stemming:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(w) for w in tokens]

    if use_lemmatization:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(w, pos='v') for w in tokens]

    return tokens


# Apply the pipeline to the entire corpus and display before/after for the first 5 sentences
preprocessed_corpus = [full_preprocess(doc) for doc in raw_corpus]

print("Preprocessing applied to the whole corpus (showing first 5 documents):\n")
for i in range(5):
    print(f"Original     [{i+1}]:", raw_corpus[i])
    print(f"Preprocessed [{i+1}]:", preprocessed_corpus[i])
    print()

Preprocessing applied to the whole corpus (showing first 5 documents):

Original     [1]: The quick brown fox jumps over the lazy dog.
Preprocessed [1]: ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']

Original     [2]: A lazy dog sleeps in the warm sun all afternoon.
Preprocessed [2]: ['lazy', 'dog', 'sleep', 'warm', 'sun', 'afternoon']

Original     [3]: Foxes are quick, clever, and highly adaptable animals.
Preprocessed [3]: ['fox', 'quick', 'clever', 'highly', 'adaptable', 'animals']

Original     [4]: The dog barks loudly whenever a stranger walks by.
Preprocessed [4]: ['dog', 'bark', 'loudly', 'whenever', 'stranger', 'walk']

Original     [5]: Cats and dogs often become the best of friends.
Preprocessed [5]: ['cat', 'dog', 'often', 'become', 'best', 'friends']



## 6. Bag of Words (BoW) (Lab 2)

The **Bag of Words** model represents each document as a vector of raw word counts,
discarding grammar and word order entirely. Given a vocabulary `V = {v1, ..., vN}`
extracted from the corpus `D`, a document `d` is represented as:

`x_d = [ f(v1, d), f(v2, d), ..., f(vN, d) ]`

where `f(v_j, d)` is the raw count of vocabulary word `v_j` in document `d`.

We build the BoW representation using `sklearn`'s `CountVectorizer` on the preprocessed
corpus.

In [6]:
def bag_of_words(token_docs):
    """
    Builds a Bag-of-Words representation from a list of tokenized documents.
    Returns: vocabulary (list), the fitted vectorizer, and the dense count matrix.
    """
    # CountVectorizer expects space-separated strings, not token lists
    joined_docs = [" ".join(doc) for doc in token_docs]

    vectorizer = CountVectorizer()
    count_matrix = vectorizer.fit_transform(joined_docs)

    vocabulary = vectorizer.get_feature_names_out()
    return vocabulary, vectorizer, count_matrix.toarray()


bow_vocab, bow_vectorizer, bow_matrix = bag_of_words(preprocessed_corpus)

print(f"Vocabulary size: {len(bow_vocab)}")
print("Vocabulary      :", list(bow_vocab))
print()

# Display the full count matrix as a labelled DataFrame
bow_df = pd.DataFrame(
    bow_matrix,
    columns=bow_vocab,
    index=[f"Doc {i+1}" for i in range(len(preprocessed_corpus))]
)
print("Bag of Words - Document Vectors (Count Matrix):")
bow_df

Vocabulary size: 93
Vocabulary      : ['across', 'adaptable', 'afternoon', 'ancient', 'animals', 'autumn', 'bark', 'bear', 'become', 'beneath', 'best', 'bird', 'breeze', 'bright', 'brown', 'calm', 'cat', 'children', 'clever', 'cold', 'cover', 'crop', 'deep', 'dog', 'early', 'fall', 'farmer', 'farmers', 'fell', 'field', 'flow', 'forest', 'fox', 'friends', 'full', 'gentle', 'gently', 'golden', 'green', 'grind', 'grow', 'happily', 'hard', 'high', 'highly', 'howl', 'jump', 'lazy', 'leave', 'life', 'light', 'little', 'loudly', 'moon', 'morning', 'mountains', 'move', 'near', 'night', 'often', 'old', 'onto', 'open', 'outsmart', 'play', 'quick', 'quiet', 'rain', 'rise', 'river', 'rivers', 'rooftops', 'silver', 'sing', 'sleep', 'sleepy', 'slowly', 'softly', 'stranger', 'summer', 'sun', 'sweetly', 'tall', 'tend', 'valley', 'village', 'wake', 'walk', 'wander', 'warm', 'whenever', 'wolves', 'wooden']

Bag of Words - Document Vectors (Count Matrix):


,across,adaptable,afternoon,ancient,animals,autumn,bark,bear,become,beneath,best,bird,breeze,bright,brown,calm,cat,children,clever,cold,cover,crop,deep,dog,early,fall,farmer,farmers,fell,field,flow,forest,fox,friends,full,gentle,gently,golden,green,grind,grow,happily,hard,high,highly,howl,jump,lazy,leave,life,light,little,loudly,moon,morning,mountains,move,near,night,often,old,onto,open,outsmart,play,quick,quiet,rain,rise,river,rivers,rooftops,silver,sing,sleep,sleepy,slowly,softly,stranger,summer,sun,sweetly,tall,tend,valley,village,wake,walk,wander,warm,whenever,wolves,wooden
Doc 1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0
Doc 3,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 4,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0
Doc 5,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 7,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
Doc 8,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
Doc 9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 10,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


**Interpretation:** each row of the table above is one sentence from the corpus,
represented purely as counts of vocabulary words it contains. Row order and word order
inside a sentence are lost — only frequency survives. Notice that common thematic words
(e.g. *forest*, *dog*, *fox*) receive higher counts because the corpus was built around
a nature theme, which is exactly the kind of signal BoW captures.

## 7. TF-IDF (Lab 2)

**TF-IDF** improves on BoW by down-weighting words that occur in *many* documents
(and are therefore less discriminative) and up-weighting words that are rarer and more
informative.

`TF(t, d) = count(t, d) / total words in d`

`IDF(t, D) = log( |D| / (1 + |{d in D : t in d}|) )`

`TF-IDF(t, d, D) = TF(t, d) x IDF(t, D)`

(`sklearn`'s `TfidfVectorizer` uses a smoothed variant of this formula internally and
L2-normalizes each row by default.)

In [7]:
def tfidf_representation(token_docs):
    """
    Builds a TF-IDF representation from a list of tokenized documents.
    Returns: vocabulary (list), the fitted vectorizer, and the dense TF-IDF matrix.
    """
    joined_docs = [" ".join(doc) for doc in token_docs]

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(joined_docs)

    vocabulary = vectorizer.get_feature_names_out()
    return vocabulary, vectorizer, tfidf_matrix.toarray()


tfidf_vocab, tfidf_vectorizer, tfidf_matrix = tfidf_representation(preprocessed_corpus)

print(f"Vocabulary size: {len(tfidf_vocab)}")
print("Vocabulary      :", list(tfidf_vocab))
print()

tfidf_df = pd.DataFrame(
    np.round(tfidf_matrix, 3),
    columns=tfidf_vocab,
    index=[f"Doc {i+1}" for i in range(len(preprocessed_corpus))]
)
print("TF-IDF Matrix (rounded to 3 decimals):")
tfidf_df

Vocabulary size: 93
Vocabulary      : ['across', 'adaptable', 'afternoon', 'ancient', 'animals', 'autumn', 'bark', 'bear', 'become', 'beneath', 'best', 'bird', 'breeze', 'bright', 'brown', 'calm', 'cat', 'children', 'clever', 'cold', 'cover', 'crop', 'deep', 'dog', 'early', 'fall', 'farmer', 'farmers', 'fell', 'field', 'flow', 'forest', 'fox', 'friends', 'full', 'gentle', 'gently', 'golden', 'green', 'grind', 'grow', 'happily', 'hard', 'high', 'highly', 'howl', 'jump', 'lazy', 'leave', 'life', 'light', 'little', 'loudly', 'moon', 'morning', 'mountains', 'move', 'near', 'night', 'often', 'old', 'onto', 'open', 'outsmart', 'play', 'quick', 'quiet', 'rain', 'rise', 'river', 'rivers', 'rooftops', 'silver', 'sing', 'sleep', 'sleepy', 'slowly', 'softly', 'stranger', 'summer', 'sun', 'sweetly', 'tall', 'tend', 'valley', 'village', 'wake', 'walk', 'wander', 'warm', 'whenever', 'wolves', 'wooden']

TF-IDF Matrix (rounded to 3 decimals):


,across,adaptable,afternoon,ancient,animals,autumn,bark,bear,become,beneath,best,bird,breeze,bright,brown,calm,cat,children,clever,cold,cover,crop,deep,dog,early,fall,farmer,farmers,fell,field,flow,forest,fox,friends,full,gentle,gently,golden,green,grind,grow,happily,hard,high,highly,howl,jump,lazy,leave,life,light,little,loudly,moon,morning,mountains,move,near,night,often,old,onto,open,outsmart,play,quick,quiet,rain,rise,river,rivers,rooftops,silver,sing,sleep,sleepy,slowly,softly,stranger,summer,sun,sweetly,tall,tend,valley,village,wake,walk,wander,warm,whenever,wolves,wooden
Doc 1,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.416,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.343,0.000,0.00,0.000,0.000,0.000,0.000,0.00,0.000,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.000,0.473,0.416,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.416,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000
Doc 2,0.000,0.00,0.444,0.00,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.323,0.000,0.00,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.390,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.444,0.000,0.000,0.000,0.000,0.000,0.390,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.444,0.000,0.000,0.000
Doc 3,0.000,0.44,0.000,0.00,0.44,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.386,0.00,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.00,0.000,0.349,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.44,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.386,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000
Doc 4,0.000,0.00,0.000,0.00,0.00,0.000,0.425,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.309,0.000,0.00,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.425,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.425,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.425,0.00,0.000,0.425,0.000,0.000
Doc 5,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.00,0.425,0.000,0.425,0.000,0.000,0.000,0.000,0.000,0.425,0.000,0.000,0.00,0.000,0.000,0.00,0.309,0.000,0.00,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.425,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.425,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000
Doc 6,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.477,0.00,0.000,0.000,0.00,0.000,0.000,0.00,0.542,0.000,0.000,0.000,0.00,0.000,0.430,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.542,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000
Doc 7,0.000,0.00,0.000,0.00,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.425,0.000,0.000,0.000,0.000,0.000,0.000,0.000

**Interpretation:** compare this table with the BoW table above. Words that appear
in almost every sentence (if any survived stop-word removal) get *pulled down* toward
zero, while words unique to only one or two sentences keep a relatively high weight.
This is why TF-IDF vectors are generally preferred over raw BoW counts for tasks like
document similarity and information retrieval.

## 8. Edit Distance (Levenshtein) (Lab 1)

The **Levenshtein Edit Distance** between two strings is the minimum number of single
character **insertions**, **deletions**, and **substitutions** required to transform one
string into the other. Given strings `A` (length `m`) and `B` (length `n`):

```
D(i, j) = max(i, j)                                        if min(i, j) == 0
D(i, j) = min( D(i-1, j) + 1,        # deletion
               D(i, j-1) + 1,        # insertion
               D(i-1, j-1) + cost )  # substitution
where cost = 0 if A[i] == B[j] else 1
```

We compute this with **dynamic programming**, filling an `(m+1) x (n+1)` matrix bottom-up.

In [8]:
def edit_distance(word1, word2, show_matrix=False):
    """
    Computes the Levenshtein edit distance between word1 and word2 using
    bottom-up dynamic programming. Optionally prints the full DP matrix.
    """
    m, n = len(word1), len(word2)
    dp = np.zeros((m + 1, n + 1), dtype=int)

    # Base cases: transforming to/from an empty string costs i (or j) insertions/deletions
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if word1[i - 1] == word2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,       # deletion
                dp[i][j - 1] + 1,       # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )

    if show_matrix:
        labels_cols = [''] + list(word2)
        labels_rows = [''] + list(word1)
        matrix_df = pd.DataFrame(dp, index=labels_rows, columns=labels_cols)
        print("Dynamic Programming Matrix:")
        print(matrix_df)
        print()

    return dp[m][n]


# Demonstration
demo_dist = edit_distance("Kitten", "Sitting", show_matrix=True)
print(f"Edit distance between 'Kitten' and 'Sitting' = {demo_dist}")

Dynamic Programming Matrix:
      S  i  t  t  i  n  g
   0  1  2  3  4  5  6  7
K  1  1  2  3  4  5  6  7
i  2  2  1  2  3  4  5  6
t  3  3  2  1  2  3  4  5
t  4  4  3  2  1  2  3  4
e  5  5  4  3  2  2  3  4
n  6  6  5  4  3  3  2  3

Edit distance between 'Kitten' and 'Sitting' = 3


## 9. Spelling Correction (Lab 1)

Using `edit_distance()` as a building block, we implement a simple dictionary-based
spelling corrector: given a misspelled word and a small vocabulary of correct words, we
compute the edit distance to every vocabulary entry, display all distances, and predict
the closest (minimum-distance) match.

Try changing `misspelled_word` below to any word to test the corrector interactively.

In [9]:
def spelling_corrector(word, vocab):
    """
    Corrects `word` by finding the vocabulary entry with the smallest edit distance.
    Prints the full distance table and returns the best suggestion.
    """
    distances = {valid_word: edit_distance(word, valid_word) for valid_word in vocab}

    distance_df = pd.DataFrame(
        sorted(distances.items(), key=lambda item: item[1]),
        columns=["Vocabulary Word", "Edit Distance"]
    )
    print(f"Edit distances from '{word}' to every vocabulary word:")
    print(distance_df.to_string(index=False))

    best_suggestion = distance_df.iloc[0]["Vocabulary Word"]
    return best_suggestion


# A small vocabulary ("dictionary") of correctly spelled words
vocabulary = ["apple", "banana", "orange", "grape", "strawberry", "pineapple", "mango"]

# --- User input for the misspelled word ---
# In a plain .py script or interactive session you could instead write:
#   misspelled_word = input("Enter a misspelled word: ")
misspelled_word = "bananna"   # <-- change this value (or use input()) to test other words

print(f"Misspelled word entered: '{misspelled_word}'\n")
suggestion = spelling_corrector(misspelled_word, vocabulary)
print(f"\nDid you mean: '{suggestion}' instead of '{misspelled_word}'?")

Misspelled word entered: 'bananna'

Edit distances from 'bananna' to every vocabulary word:
Vocabulary Word  Edit Distance
         banana              1
         orange              5
          mango              5
          apple              6
          grape              6
      pineapple              7
     strawberry              9

Did you mean: 'banana' instead of 'bananna'?


## 10. Sentence Boundary Creation (Lab 2)

Before we can train an N-gram language model, every sentence in the corpus must be
explicitly wrapped with `<s>` (sentence start) and `</s>` (sentence end) tokens. These
boundary markers let the model learn:

* which words are likely to **start** a sentence (via histories containing `<s>`), and
* which words are likely to **end** a sentence (via `</s>` as a predicted "next word").

Boundary tokens must **never** be stripped out by the preprocessing pipeline (unlike
ordinary punctuation), so we add them *after* cleaning each sentence, not before.

In [10]:
def add_sentence_boundaries(sentences):
    """
    Cleans each raw sentence (lowercase, remove punctuation/numbers, collapse spaces,
    tokenize, WITHOUT removing stopwords so natural sentence flow is preserved) and
    wraps the resulting token list with <s> ... </s> boundary markers.
    """
    bounded_sentences = []
    for sent in sentences:
        tokens = full_preprocess(sent, remove_stopwords=False, use_lemmatization=False)
        bounded_sentences.append(["<s>"] + tokens + ["</s>"])
    return bounded_sentences


bounded_corpus = add_sentence_boundaries(raw_corpus)

print("Corpus with sentence boundaries (first 5 sentences):\n")
for i in range(5):
    print(f"{i+1:2d}. " + " ".join(bounded_corpus[i]))

# Flatten into one long token stream for N-gram training, sentence by sentence
all_tokens = [tok for sent in bounded_corpus for tok in sent]
print(f"\nTotal tokens across the whole boundary-tagged corpus: {len(all_tokens)}")
print("First 20 tokens of the flattened stream:", all_tokens[:20])

Corpus with sentence boundaries (first 5 sentences):

 1. <s> the quick brown fox jumps over the lazy dog </s>
 2. <s> a lazy dog sleeps in the warm sun all afternoon </s>
 3. <s> foxes are quick clever and highly adaptable animals </s>
 4. <s> the dog barks loudly whenever a stranger walks by </s>
 5. <s> cats and dogs often become the best of friends </s>

Total tokens across the whole boundary-tagged corpus: 211
First 20 tokens of the flattened stream: ['<s>', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '</s>', '<s>', 'a', 'lazy', 'dog', 'sleeps', 'in', 'the', 'warm', 'sun']


## 11. Dynamic N-Gram Language Model (Lab 2)

### 11.1 Mathematical Background

An **N-gram** is a contiguous sequence of `N` tokens from a text.

* `N = 1` → **Unigram** (e.g. `"fox"`)
* `N = 2` → **Bigram** (e.g. `"the fox"`)
* `N = 3` → **Trigram** (e.g. `"the quick fox"`)
* `N = 4` → **4-gram**, and so on.

**Chain Rule of Probability.** The joint probability of a word sequence
`W = (w1, w2, ..., wk)` is, by the chain rule:

`P(w1, ..., wk) = Product over i=1..k of P(wi | w1, ..., w(i-1))`

**Markov Assumption.** Conditioning on the *entire* history is infeasible, so we assume a
word depends only on the previous `N-1` words:

`P(wi | w1, ..., w(i-1)) ~= P(wi | w(i-N+1), ..., w(i-1))`

* Unigram (`N=1`): `P(wi) = C(wi) / sum_w C(w)`
* Bigram  (`N=2`): `P(wi | w(i-1)) = C(w(i-1), wi) / C(w(i-1))`
* Trigram (`N=3`): `P(wi | w(i-2), w(i-1)) = C(w(i-2), w(i-1), wi) / C(w(i-2), w(i-1))`
* General N-gram : `P(wi | w(i-N+1..i-1)) = C(w(i-N+1..i-1), wi) / C(w(i-N+1..i-1))`

**Maximum Likelihood Estimation (MLE).** All of the formulas above are estimated
directly from corpus counts — this *is* MLE for N-gram models: the probability of the
next word given a history is simply how often that (history, next-word) pair was
observed, divided by how often that history was observed on its own.

### 11.2 Dynamic Implementation

Crucially, the function below is **not** hard-coded for unigrams or bigrams — the
history window size is always `n - 1`, so passing `n=1, 2, 3, 4, ...` automatically
produces the correct model with no code changes.

In [11]:
def build_ngram_counts(tokens, n):
    """
    Builds raw N-gram counts from a flat token stream, for ANY n >= 1.

    For n == 1 (unigrams), history is the empty tuple () for every token, and the
    'next word' is the token itself -- this naturally falls out of the same sliding
    window logic used for n >= 2, so no special-casing is required.

    Returns: a defaultdict(Counter) mapping history-tuple -> Counter(next_word -> count)
    """
    model = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i:i + n - 1])   # empty tuple when n == 1
        next_word = tokens[i + n - 1]
        model[history][next_word] += 1
    return model


def build_probability_table(ngram_counts):
    """
    Converts raw N-gram counts into MLE probabilities.
    Returns: defaultdict(dict) mapping history-tuple -> {next_word: probability}
    """
    prob_model = defaultdict(dict)
    for history, next_word_counts in ngram_counts.items():
        history_total = sum(next_word_counts.values())   # C(history)
        for next_word, count in next_word_counts.items():
            prob_model[history][next_word] = count / history_total
    return prob_model


def train_ngram_model(tokens, n):
    """Convenience wrapper: builds counts AND the MLE probability table together."""
    counts = build_ngram_counts(tokens, n)
    probabilities = build_probability_table(counts)
    return counts, probabilities


print("build_ngram_counts(), build_probability_table(), and train_ngram_model() defined.")
print("These work identically for ANY value of n -- no hardcoded unigram/bigram logic.")

build_ngram_counts(), build_probability_table(), and train_ngram_model() defined.
These work identically for ANY value of n -- no hardcoded unigram/bigram logic.


## 12. N-Gram Count Tables

### Choose N

Below, set `N` to any positive integer (1 = unigram, 2 = bigram, 3 = trigram, 4 = 4-gram,
...). Everything downstream — count tables, probability tables, next-word prediction, and
sentence generation — is driven dynamically by this single value.

In [18]:
# --- User input for N ---
# In a plain .py script or interactive session you could instead write:
N = int(input("Enter N (1=unigram, 2=bigram, 3=trigram, 4=4-gram, ...): "))
#N = 2   # <-- change this value (or use input()) to try N = 1, 3, 4, ...

print(f"Training a {N}-gram model on the boundary-tagged corpus (N = {N})...")
ngram_counts, ngram_probabilities = train_ngram_model(all_tokens, N)
print(f"Number of distinct histories learned: {len(ngram_counts)}")

Training a 4-gram model on the boundary-tagged corpus (N = 4)...
Number of distinct histories learned: 198


In [19]:
def display_count_table(ngram_counts, n):
    """
    Renders the N-gram count dictionary as a pandas DataFrame.
    Column layout adapts automatically to n:
        n == 1 -> Word | Count
        n >= 2 -> History | Next Word | Count | History Count
    """
    rows = []
    if n == 1:
        for _, next_word_counts in ngram_counts.items():
            for word, count in next_word_counts.items():
                rows.append({"Word": word, "Count": count})
        df = pd.DataFrame(rows).sort_values("Count", ascending=False).reset_index(drop=True)
    else:
        for history, next_word_counts in ngram_counts.items():
            history_total = sum(next_word_counts.values())
            history_str = " ".join(history)
            for next_word, count in next_word_counts.items():
                rows.append({
                    "History": history_str,
                    "Next Word": next_word,
                    "Count": count,
                    "History Count": history_total
                })
        df = pd.DataFrame(rows).sort_values(["History", "Count"], ascending=[True, False]).reset_index(drop=True)
    return df


count_table = display_count_table(ngram_counts, N)
print(f"{N}-Gram Count Table ({len(count_table)} rows):")
count_table

4-Gram Count Table (207 rows):


,History,Next Word,Count,History Count
0,</s> <s> a,lazy,1,3
1,</s> <s> a,brown,1,3
2,</s> <s> a,gentle,1,3
3,</s> <s> autumn,leaves,1,1
4,</s> <s> birds,sing,1,1
...,...,...,...,...
202,was covered in,golden,1,1
203,whenever a stranger,walks,1,1
204,with silver light,</s>,1,1
205,wolves howl at,night,1,1


## 13. MLE Probability Tables

We now generate the Maximum Likelihood Estimation probability table for the same `N`,
following the formula from Section 11.1:

`P(next word | history) = C(history, next word) / C(history)`

For each row: `History`, `Next Word`, `Count(history, next word)`, `Count(history)`,
and the resulting `Probability`.

In [20]:
def display_probability_table(ngram_counts, n):
    """
    Renders the MLE probability table as a pandas DataFrame, matching the exact
    formula P(next word | history) = C(history, next word) / C(history).
    Column layout adapts automatically to n (unigram has no 'history' column).
    """
    rows = []
    if n == 1:
        total = sum(sum(c.values()) for c in ngram_counts.values())
        for _, next_word_counts in ngram_counts.items():
            for word, count in next_word_counts.items():
                rows.append({
                    "Word": word,
                    "Count": count,
                    "Total Tokens": total,
                    "Probability": count / total
                })
        df = pd.DataFrame(rows).sort_values("Probability", ascending=False).reset_index(drop=True)
    else:
        for history, next_word_counts in ngram_counts.items():
            history_total = sum(next_word_counts.values())
            history_str = " ".join(history)
            for next_word, count in next_word_counts.items():
                rows.append({
                    "History": history_str,
                    "Next Word": next_word,
                    "Count(history, next word)": count,
                    "Count(history)": history_total,
                    "Probability": count / history_total
                })
        df = pd.DataFrame(rows).sort_values(["History", "Probability"], ascending=[True, False]).reset_index(drop=True)
    return df


probability_table = display_probability_table(ngram_counts, N)
print(f"{N}-Gram MLE Probability Table ({len(probability_table)} rows):")
probability_table.round(3)

4-Gram MLE Probability Table (207 rows):


,History,Next Word,"Count(history, next word)",Count(history),Probability
0,</s> <s> a,lazy,1,3,0.333
1,</s> <s> a,brown,1,3,0.333
2,</s> <s> a,gentle,1,3,0.333
3,</s> <s> autumn,leaves,1,1,1.000
4,</s> <s> birds,sing,1,1,1.000
...,...,...,...,...,...
202,was covered in,golden,1,1,1.000
203,whenever a stranger,walks,1,1,1.000
204,with silver light,</s>,1,1,1.000
205,wolves howl at,night,1,1,1.000


**Sanity check:** for every distinct history, the probabilities of all of its
possible next words should sum to (approximately) `1.00`, since MLE simply normalizes
the observed counts. You can verify this by grouping the table above by `History` and
summing `Probability`.

In [21]:
if N > 1:
    check = probability_table.groupby("History")["Probability"].sum().round(3)
    print("Probability mass per history (should all be 1.0):")
    print(check)
else:
    print("Total probability mass across all unigrams (should be 1.0):",
          round(probability_table["Probability"].sum(), 3))

Probability mass per history (should all be 1.0):
History
</s> <s> a              1.0
</s> <s> autumn         1.0
</s> <s> birds          1.0
</s> <s> cats           1.0
</s> <s> farmers        1.0
                       ... 
was covered in          1.0
whenever a stranger     1.0
with silver light       1.0
wolves howl at          1.0
wooden rooftops </s>    1.0
Name: Probability, Length: 198, dtype: float64


## 14. Next Word Prediction — Shannon's Guessing Game

Claude Shannon's classic 1951 experiment tests how well a language model can guess the
next word given some preceding context. We replicate it here: the user supplies a phrase,
we preprocess it (keeping stop words, since word order/function words matter for the
model's history lookup), extract the trailing `N-1` words as the history key, and look up
every candidate next word the trained model has ever seen after that history — sorted by
descending probability.

In [22]:
def predict_next_word(history_phrase, prob_model, n):
    """
    Shannon's Guessing Game: given a phrase and a trained n-gram probability model,
    predicts the most probable next word(s).

    Returns a pandas DataFrame of (Next Word, Count, Probability) sorted descending
    by probability, or an empty DataFrame if the history was never observed.
    """
    # Preprocess the phrase the same way training tokens were prepared (no stopword removal,
    # so the trailing context matches the corpus's natural word order)
    processed = full_preprocess(history_phrase, remove_stopwords=False, use_lemmatization=False)

    context_size = n - 1
    history_key = tuple(processed[-context_size:]) if context_size > 0 else tuple()

    print(f"Input phrase          : '{history_phrase}'")
    print(f"Preprocessed tokens   : {processed}")
    print(f"Lookup history (n-1={context_size}): {history_key}")

    if history_key not in prob_model:
        print(f"\nNo predictions available -- this history was never observed in training for N={n}.")
        return pd.DataFrame(columns=["Next Word", "Probability"])

    candidates = prob_model[history_key]
    counts_for_history = ngram_counts[history_key]

    rows = [
        {"Next Word": word, "Count": counts_for_history[word], "Probability": prob}
        for word, prob in candidates.items()
    ]
    result_df = pd.DataFrame(rows).sort_values("Probability", ascending=False).reset_index(drop=True)
    return result_df


# --- User input for the seed phrase ---
# In a plain .py script or interactive session you could instead write:
#   input_phrase = input("Enter a phrase: ")
input_phrase = "the lazy" if N > 1 else "the"   # <-- change this to test other phrases

print("=== Shannon's Guessing Game ===\n")
predictions = predict_next_word(input_phrase, ngram_probabilities, N)
print()
predictions.round(3)

=== Shannon's Guessing Game ===

Input phrase          : 'the lazy'
Preprocessed tokens   : ['the', 'lazy']
Lookup history (n-1=3): ('the', 'lazy')

No predictions available -- this history was never observed in training for N=4.



,Next Word,Probability


## 15. Sentence Generation

Finally, we use the trained N-gram model *generatively*: starting from a seed phrase, we
repeatedly sample the next word from the model's probability distribution over the
current history (using `np.random.choice`, weighted by probability — **not** always
picking the single most likely word, which keeps the output varied and natural). Generation
stops when the model produces `</s>` or when `max_length` words have been generated.

In [23]:
def generate_sentence(seed_phrase, prob_model, n, max_length=20, random_seed=None):
    """
    Probabilistically generates a sentence from a trained n-gram model.

    - seed_phrase : starting text (will be preprocessed the same way as training data)
    - prob_model  : the MLE probability table from build_probability_table()
    - n           : the n-gram order the model was trained with
    - max_length  : safety cap on the number of generated tokens
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    context_size = n - 1
    tokens = full_preprocess(seed_phrase, remove_stopwords=False, use_lemmatization=False)
    if not tokens or tokens[0] != "<s>":
        tokens = ["<s>"] + tokens  # ensure generation always starts from a valid sentence-start context

    for _ in range(max_length):
        history_key = tuple(tokens[-context_size:]) if context_size > 0 else tuple()

        if history_key not in prob_model:
            break  # unseen history -- stop generating

        next_words = list(prob_model[history_key].keys())
        probabilities = list(prob_model[history_key].values())

        # Weighted random sampling: higher-probability words are more likely to be chosen
        next_word = np.random.choice(next_words, p=probabilities)
        tokens.append(next_word)

        if next_word == "</s>":
            break

    return " ".join(tokens)


# --- User input for seed phrase and max length ---
# In a plain .py script or interactive session you could instead write:
#   seed = input("Enter a seed phrase: ")
#   max_len = int(input("Enter maximum sentence length: "))
seed = "<s> the"
max_len = 15

print(f"Generating up to 5 sentences from seed = '{seed}' (N={N}, max_length={max_len})\n")
for i in range(5):
    sentence = generate_sentence(seed, ngram_probabilities, N, max_length=max_len)
    print(f"{i+1}. {sentence}")

Generating up to 5 sentences from seed = '<s> the' (N=4, max_length=15)

1. <s> the
2. <s> the
3. <s> the
4. <s> the
5. <s> the


**Note:** with a small, 20-sentence training corpus, higher-order models (N=3 or
N=4) will often have very few — sometimes only one — observed continuations for a given
history, so generation can terminate quickly or become deterministic. This is expected:
it directly illustrates the classic **data sparsity problem** in N-gram language modeling,
which is one of the main motivations for smoothing techniques (e.g. Laplace/Add-1
smoothing, Kneser-Ney) and, eventually, neural language models.

## 16. Conclusion

This notebook implemented the complete pipeline taught across Lab 1 and Lab 2:

* **Preprocessing:** raw text was systematically cleaned via regex (HTML/number/symbol
  removal, whitespace normalization), tokenized, filtered of stop words, and normalized
  via both stemming and lemmatization.
* **Vector representations:** the same cleaned corpus was converted into both **Bag of
  Words** count vectors and **TF-IDF** weighted vectors, and the two were compared.
* **Edit distance & spelling correction:** a dynamic-programming Levenshtein implementation
  powered a simple dictionary-based spell checker.
* **N-gram language modeling:** sentence-boundary-tagged text was used to train a fully
  **dynamic** N-gram model (any N), from which we derived count tables, MLE probability
  tables, next-word predictions (Shannon's Guessing Game), and probabilistic sentence
  generation.

Together, these techniques form the classical statistical NLP toolkit that underpins
search engines, spell checkers, autocomplete systems, and — as simplified precursors —
the large language models used today.